In [11]:
# Gianna Miller worked for 11 hours,
#imported necessary packages after researching which ones were useful,
#created pathway to ensure new user file for each user,
#made the code pull up the old user information, created the login interface
#and app interface, created drop down for selecting habits, created input
#interface to put in habits for each habit, created the graphs for the tracked
#habits, created the login interface to ensure user had an account and a unique
#username and trialed the app

# import necessary packages
import pandas as pd
from datetime import date
import os
import hashlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# create new file for each user
user_file = "user.csv"
data_folder = "user_data"
os.makedirs(data_folder, exist_ok = True)

# create habit data for first use or load data after first use
if os.path.isfile(user_file):
  users = pd.read_csv(user_file)
else:
  users = pd.DataFrame(columns = ["Username", "PasswordHash"])
  users.to_csv(user_file, index = False)

current_user = None

# save password
def hash_password(password):
  return hashlib.sha256(password.encode()).hexdigest()

# load user data
def get_user_file(username):
  return f"{data_folder}/{username}_habits.csv"
def load_user_data(username):
  user_file = get_user_file(username)
  if os.path.exists(user_file):
    return pd.read_csv(user_file)
  else:
    return pd.DataFrame(columns = ["Date", "Habit", "Value", "Unit", "Notes"])
def save_user_data(username, data):
  data.to_csv(get_user_file(username), index = False)

# Login interface
title = widgets.HTML("<h1>Habits App</h1>")
username_box = widgets.Text(description = "Username: ")
password_box = widgets.Password(description = "Password: ")
login_button = widgets.Button(description = "Login", button_style = "success")
signup_button = widgets.Button(description = "Sign Up", button_style = "info")
login_output = widgets.Output()

# interface for right after logged in
def show_app(username):
  clear_output()
  habits = load_user_data(username)
  welcome = widgets.HTML(f"<h2>Welcome, {username}!</h2>")

  # create a habit dropdown list
  habit_dropdown = widgets.Dropdown(options = ["Water Consumption", "Pages Read", "Productivity", "Sleep Consistency"], description = "Habit: ")
  input_area = widgets.VBox()

  # create save, view, summary, graph, and logout button that is universal to all habits
  save_button = widgets.Button(description = "Save Entry", button_style = "success")
  view_button = widgets.Button(description = "View My Data", button_style = "info")
  summary_button = widgets.Button(description = "Show Summary", button_style = "warning")
  graph_button = widgets.Button(description = "Show Graphs", button_style = "primary")
  logout_button = widgets.Button(description = "Logout", button_style = "danger")
  output = widgets.Output()

  # create places for the user to input habits
  def update_inputs(change):
    selected = habit_dropdown.value
    if selected == "Water Consumption":
      ounces_box = widgets.FloatSlider(value = 8, min = 0, max = 30, step = 1, description = "Ounces: ")
      notes_box = widgets.Textarea(description = "Notes: ", placeholder = "Optional Notes")
      input_area.children = [ounces_box, notes_box]

    elif selected == "Pages Read":
      pages_box = widgets.IntSlider(value = 30, min = 0, max = 300, step = 1, description = "Pages: ")
      book_box = widgets.Text(description = "Book: ", placeholder = "Book Title")
      input_area.children = [pages_box, book_box]

    elif selected == "Productivity":
      hours_box = widgets.FloatSlider(value = 2, min = 0, max = 16, step = 0.5, description = "Hours: ")
      task_box = widgets.Textarea(description = "Tasks: ", placeholder = "What did you work on?")
      input_area.children = [hours_box, task_box]

    elif selected == "Sleep Consistency":
      sleep_box = widgets.FloatSlider(value = 8, min = 0, max = 14, step = 0.5, description = "Hours: ")
      bedtime_box = widgets.Text(description = "Bedtime: ", placeholder = "Example: 10:00 PM")
      wake_box = widgets.Text(description = "Wake Time: ", placeholder = "Example: 7:00 AM")
      input_area.children = [sleep_box, bedtime_box, wake_box]

  # save the inputted information
  def save_entry(button):
    nonlocal habits
    selected = habit_dropdown.value
    if selected == "Water Consumption":
      value = input_area.children[0].value
      unit = "ounces"
      notes = input_area.children[1].value
    elif selected == "Pages Read":
      value = input_area.children[0].value
      unit = "pages"
      notes = f"Book: {input_area.children[1].value}"
    elif selected == "Productivity":
      value = input_area.children[0].value
      unit = "hours"
      notes = input_area.children[1].value
    elif selected == "Sleep Consistency":
      value = input_area.children[0].value
      unit = "hours"
      notes = (f"Bedtime: {input_area.children[1].value}, " f"Wake Time: {input_area.children[2].value}")
    new_entry = {"Date": str(date.today()), "Habit": selected, "Value": value, "Unit": unit, "Notes": notes}
    habits = pd.concat([habits, pd.DataFrame([new_entry])], ignore_index = True)
    save_user_data(username, habits)
    with output:
      clear_output()
      print("Entry saved!")
      display(habits.tail(1))

  # show the habit data
  def view_data(button):
    with output:
      clear_output()
      if len(habits) == 0:
        print("No data yet.")
      else:
        display(habits)
  # display a summary of the habits
  def show_summary(button):
    with output:
      clear_output()
      if len(habits) == 0:
        print("No data yet.")
      else:
        summary = habits.groupby("Habit")["Value"].mean()
        display(summary)
  # Show the graphs of their habits
  def show_graphs(button):
    with output:
      clear_output()
      if len(habits) == 0:
        print("No data yet.")
        return
      data = habits.copy()
      data["Date"] = pd.to_datetime(data["Date"])
      for habit in data["Habit"].unique():
        habit_data = data[data["Habit"] == habit]
        plt.figure()
        plt.plot(habit_data["Date"], habit_data["Value"], marker = "o")
        plt.xlabel("Date")
        plt.ylabel("Value")
        plt.title(f"{habit} Trend")
        plt.xticks(rotation = 45)
        plt.grid(True)
        plt.show()

  def logout(button):
    clear_output()
    display_login()

  # save it
  habit_dropdown.observe(update_inputs, names = "value")
  update_inputs(None)
  save_button.on_click(save_entry)
  view_button.on_click(view_data)
  summary_button.on_click(show_summary)
  graph_button.on_click(show_graphs)
  logout_button.on_click(logout)

  # save all of this into the app
  app = widgets.VBox([welcome, habit_dropdown, input_area, widgets.HBox([save_button, view_button, summary_button, graph_button, logout_button]), output])
  display(app)

# have the login make sure user exists or signup and have unique usernames for every person
def login(button):
  global users
  username = username_box.value.strip()
  password = password_box.value
  with login_output:
    clear_output()
    if username == "" or password == "":
      print("Enter a username and password.")
      return
    password_hash = hash_password(password)
    match = users[(users["Username"] == username) & (users["PasswordHash"] == password_hash)]

    if len(match) == 1:
      show_app(username)
    else:
      print("Incorrect username or password.")

def signup(button):
  global users
  username = username_box.value.strip()
  password = password_box.value
  with login_output:
    clear_output
    if username == "" or password == "":
      print("Enter a username and password.")
      return
    if username in users["Username"].values:
      print("Username already exists.")
      return
    new_user = {"Username": username, "PasswordHash": hash_password(password)}
    users = pd.concat([users, pd.DataFrame([new_user])], ignore_index = True)
    users.to_csv(user_file, index = False)
    empty_data = pd.DataFrame(columns = ["Date", "Habit", "Value", "Unit", "Notes"])
    save_user_data(username, empty_data)
    print("Account created. You may now log in.")

def display_login():
  login_button.on_click(login)
  signup_button.on_click(signup)
  login_page = widgets.VBox([title, username_box, password_box, widgets.HBox([login_button, signup_button]), login_output])
  display(login_page)

display_login()



/tmp/ipykernel_6600/2179325841.py:112: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  habits = pd.concat([habits, pd.DataFrame([new_entry])], ignore_index = True)
